# Quantum walks with structured block encodings

Verify the extended tree, circulant Chebyshev response, and both full marked-vertex walks.

## Identify the structured Circuit work behind the blocks

The current implementation constructs tree, circulant and discriminant walk
blocks with explicit Circuit gates and uses native synth for the Chebyshev
input preparation. The matrices, reflection geometry and address maps remain
user-supplied algorithms, not structure automatically discovered by synth.
The original marked and unmarked walks are both retained.

Try comparing the zero-selector block against the declared matrix before
normalizing its state. Report scale, absolute error and acceptance probability
separately; successful state normalization cannot verify a block encoding.
The Chebyshev target is the declared matrix polynomial, while local gate
resources describe the supplied implementation rather than a general optimum.


Set up the local FlagQuantum experiments.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.community.paper_implementation_project.explicit_quantum_circuits_for_block_encoding.quantum_walks_via_efficient_blockencoding.overview import (
    model,
    quantum,
    workflow,
)
from examples.synthesis.community.paper_implementation_project.explicit_quantum_circuits_for_block_encoding.quantum_walks_via_efficient_blockencoding.overview.main import (
    print_report,
    save_figures,
)


Build the original eight-vertex extended tree adjacency matrix.

In [ ]:
print(model.tree(8))


Use the original edge, self-loop and root rotation coefficients.

In [ ]:
print("Tree rotations:", quantum.TREE_THETA)


Execute the complete graph workflow and check the eight-vertex structured projection.

In [ ]:
result = workflow.run()
assert result.trees[2].oracle_error < 1e-10
print("Eight-vertex tree projection:", result.trees[2].projection.real.tolist())


Regenerate all real Pauli coefficients and load each public LCU encoding.

In [ ]:
print("Tree Pauli term counts:", tuple(row.lcu_terms for row in result.trees))


Retain 2, 4, 8 and 16 vertices and measure actual Pauli LCU resources.

In [ ]:
assert tuple(row.size for row in result.trees) == workflow.TREE_SIZES
print("LCU normalizations:", tuple(row.alpha for row in result.trees))


Compare both implementations in a common gate basis.

In [ ]:
figures = Path("notebook-figures")
save_figures(result, figures)
display(Image(filename=str(figures / "tree-resources.png")))


Build the original eight-site banded circulant matrix and verify its stochastic rows.

In [ ]:
cyclic = model.cyclic()
print(cyclic)
assert bool(((cyclic.sum(dim=1) - 1).abs() < 1e-12).all())


Derive the two amplitudes for the coherent shift selector.

In [ ]:
print("Circulant rotation angles:", quantum.CYCLIC_THETA)


Form the source cyclic adder, swap and inverse adder.

In [ ]:
print("Cyclic template width:", quantum.cyclic_encoding().n_qubits)


Verify all cyclic basis columns against the independent shifted permutation.

In [ ]:
assert result.cyclic_error < 1e-10


Read the uniform circulant projection.

In [ ]:
print("Circulant projection:", result.cyclic_projection.real.tolist())


Construct the independent T2 target from the full cyclic matrix.

In [ ]:
print("Chebyshev target matrix:", 2 * (cyclic @ cyclic) - cyclic.new_ones(8).diag())


Use the original nonuniform amplitudes and apply two coherent projector reflections.

In [ ]:
print("Original Chebyshev data amplitudes:", quantum.CHEB_AMPLITUDES)
assert result.chebyshev_error < 1e-10


Read the projected result in the source global phase convention.

In [ ]:
print("Chebyshev projection:", result.chebyshev_projection.real.tolist())


Build both mathematical discriminants on four vertices.

In [ ]:
for marked in (False, True):
    print("Discriminant, marked =", marked, model.discriminant(4, marked=marked))


Execute the unmarked H and SWAP discriminant.

In [ ]:
unmarked, marked_walk = result.walks
assert unmarked.block_error < 1e-5


Read the full unmarked projected vector.

In [ ]:
print("Unmarked projection:", unmarked.projection.real.tolist())


Check the original rounded controlled rotations around the marked discriminant.

In [ ]:
assert marked_walk.block_error < 1e-5


Read the marked projected vector with the source display phase.

In [ ]:
print("Marked projection:", marked_walk.projection.real.tolist())


Sample the 16-vertex walks at each of the original 20 repetitions.

In [ ]:
print_report(result)
display(Image(filename=str(figures / "walk-zero-block.png")))
